# 05 — Five Prompts in Five Minutes: The Multi-Template Pipeline

**The pitch.** A complex business question doesn't deserve one prompt — it deserves five. One to decompose it, one to analyze the data, one to assess risks, one to synthesize findings, one to adapt for the audience. Without mycontext: hours of writing and tuning. With mycontext: `suggest_patterns(question, suggest_chain=True)` recommends the chain, `suggest_routes` adds multiple route options with reasoning, `TemplateIntegratorAgent` merges all five templates into one unified prompt. Quality-gated at every step.

**Who this is for:** Developers building multi-step LLM pipelines, analysts who need structured multi-angle reasoning.

**What you'll learn:**
- `suggest_patterns(suggest_chain=True)` — get a full pipeline recommendation
- `suggest_routes` — multiple route options with reasoning
- `TemplateIntegratorAgent.suggest_and_integrate()` — merge templates into one unified prompt
- Quality-gate each step in the pipeline
- Export the integrated prompt for any framework

**Next:** **06** = Cross-framework portability. **07** = Executable skills.

## Research grounding

> **Decomposed prompting:** Complex questions answered in a single pass are consistently shallower than those answered through structured sub-question decomposition *(Press et al. 2022 — "Measuring and Narrowing the Compositionality Gap"; Khot et al. 2022 — "Decomposed Prompting")*. The five-stage pipeline (decompose → analyze → assess → synthesize → adapt) maps each stage to a cognitively distinct operation handled by a specialist template optimised for that reasoning type.

> **Integrated vs. sequential:** mycontext sprint research shows that integrated multi-template prompts (templates merged into one coherent prompt) outperform sequential pipeline calls on structured analytical tasks — fewer context switches, a coherent shared reasoning chain, and lower total token cost than 5 separate LLM calls.

## Why five prompts beat one

Complex business questions have multiple cognitive dimensions. A single generic prompt flattens them.

| Pipeline step | Template | What it contributes |
|--------------|----------|--------------------|
| 1. Decompose | `question_analyzer` | Break the question into sub-questions |
| 2. Analyze | `data_analyzer` | Structured data/trend analysis |
| 3. Assess risk | `risk_assessor` | Identify and score risks |
| 4. Synthesize | `synthesis_builder` | Integrate findings into coherent narrative |
| 5. Adapt | `audience_adapter` | Tailor language/depth to the reader |

The integrated prompt combines all five reasoning frameworks into one — no orchestrator needed.

## Install and setup

In [1]:
# %pip install -q -U "mycontext-ai>=0.11.1"
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s, raw=True)

import mycontext

md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")


**mycontext-ai** `0.11.0`

## Step 1 — `suggest_patterns(suggest_chain=True)`: get the pipeline

In [3]:
from mycontext.intelligence import suggest_patterns

question = (
    "How will rising interest rates affect our mortgage portfolio risk, "
    "and what should we tell the board?"
)

result = suggest_patterns(
    question,
    suggest_chain=True,
    mode="keyword",
)

md("=== Suggested pipeline ===")
if result.suggested_chain:
    for i, step in enumerate(result.suggested_chain, 1):
        md(f"  Step {i}: {step}")
else:
    top_p = result.suggested_patterns[0] if result.suggested_patterns else None
    md(f"Top suggestion: {top_p.name if top_p else 'N/A'}")
    md("(suggest_chain=True returns a chain when question complexity warrants it)")


=== Suggested pipeline ===

  Step 1: risk_assessor

  Step 2: impact_assessor

  Step 3: decision_framework

In [4]:
# Also get LLM-enhanced suggestions if key is available
result_llm = suggest_patterns(
    question,
    suggest_chain=True,
    mode="hybrid",
    llm_provider="openai",
)
md("=== LLM-enhanced pipeline ===")
if result_llm.suggested_chain:
    for i, step in enumerate(result_llm.suggested_chain, 1):
        md(f"  Step {i}: {step}")
if result_llm.suggested_patterns:
    md(f"\nReason: {result_llm.suggested_patterns[0].reason}")


=== LLM-enhanced pipeline ===

  Step 1: risk_assessor

  Step 2: impact_assessor

  Step 3: decision_framework


Reason: Risk matrix with likelihood, impact, and mitigations related to interest rates on mortgage portfolio

## Step 2 — `suggest_routes`: multiple pipeline options with reasoning

In [6]:
from mycontext.intelligence import suggest_routes

routes = suggest_routes(
    question,
    max_routes=3,
    provider="openai",
)

md(f"Routes generated: {len(routes.routes)}\n")
for i, route in enumerate(routes.routes, 1):
    md(f"Route {i}: {route.label} ({route.dimension})")
    _steps = " → ".join(s.template for s in route.steps)
    md(f"  Steps: {_steps}")
    md(f"  Rationale: {(str(route.rationale)[:200-3] + '...' if len(str(route.rationale)) > 200 else str(route.rationale))}")
    md("---")


Routes generated: 3


Route 1: Predictive Impact Analysis (predictive)

  Steps: trend_identifier → scenario_planner

  Rationale: This route uses predictive analysis to forecast how rising interest rates will likely impact the mortgage portfolio, assisting financial planning and risk management.

---

...

---

Route 3: Board Communication Strategy (communication)

  Steps: audience_adapter → technical_translator

  Rationale: Effective communication with the board is critical for aligning understanding and strategies regarding financial risks and their management.

---

## Step 3 — Quality-gate each template before integration

In [7]:
from mycontext.intelligence import QualityMetrics, get_pattern_class

# Build a context for each template in our pipeline
pipeline_templates = ["question_analyzer", "risk_assessor", "synthesis_builder"]
qm = QualityMetrics(mode="heuristic")

md(f"{'Template':<28} {'Quality Score'} {'Gate (≥60)'}")
md("-" * 60)

pipeline_contexts = {}
for template_name in pipeline_templates:
    try:
        PatternClass = get_pattern_class(template_name)
        ctx = PatternClass().build_context(question=question)
        score = qm.evaluate(ctx)
        gate = "✓ PASS" if score.overall >= 0.60 else "✗ BLOCK"
        md(f"{template_name:<28} {score.overall * 100:>10.0f}/100   {gate}")
        pipeline_contexts[template_name] = ctx
    except Exception as e:
        md(f"{template_name:<28} [Error: {e}]")


Template                     Quality Score Gate (≥60)

------------------------------------------------------------

question_analyzer            [Error: QuestionAnalyzer.build_context() missing 1 required positional argument: 'self']

risk_assessor                [Error: RiskAssessor.build_context() missing 2 required positional arguments: 'self' and 'decision']

synthesis_builder            [Error: SynthesisBuilder.build_context() missing 1 required positional argument: 'self']

## Step 4 — `TemplateIntegratorAgent`: merge all templates into one unified prompt

In [11]:
from mycontext.intelligence import TemplateIntegratorAgent

agent = TemplateIntegratorAgent()

# suggest_and_integrate: pipeline suggestion + integration in one call
integrated = agent.suggest_and_integrate(
    question,
    provider="openai",
    max_patterns=5,
)

md("=== Integration result ===")
md(f"Templates integrated: {integrated.source_templates}")
md(f"\nIntegrated framework (first 1200 chars):")
ctx = integrated.to_context()
md((str(ctx.assemble())[:1200-3] + '...' if len(str(ctx.assemble())) > 1200 else str(ctx.assemble())))


=== Integration result ===

Templates integrated: ['risk_assessor', 'impact_assessor']


Integrated framework (first 1200 chars):

You are Mortgage Portfolio Risk and Impact Analyst.

Follow these rules:
1. Evaluate risks by likelihood and impact, focusing on strategic, operational, and financial categories.
2. Assess downstream effects and consequences of rising interest rates on the mortgage portfolio.
3. Prioritize risks and impacts by severity and likelihood.
4. Develop pragmatic mitigation strategies for identified risks.
5. Perform a risk-benefit analysis to inform decision-making.
6. Provide clear, actionable recommendations for the board.

CONSTRAINTS:

Must include:
  - Context Summary: Overview of rising interest rates and initial impact on mortgage portfolio.
  - Risk Identification: List and categorize risks by strategic, operational, and financial impacts.
  - Risk Analysis: Probability and impact assessment of identified risks.
  - Impact Assessment: Downstream effects on the mortgage portfolio.
  - Mitigation Strategies: Proposed actions to manage identified risks.
  - Risk-Benefit Analysis: Evaluation of potential impacts versus benefits.
  - Recommendations / Next Steps: Concrete actions for the board to consider.

1. Summarize the current context of rising interest rates and its potential ...

## Step 5 — Score the integrated prompt vs a raw prompt

In [12]:
from mycontext import Context, Guidance, Directive

raw_ctx = Context(
    guidance=Guidance(role="financial analyst", rules=["Be accurate"]),
    directive=Directive(content=question),
)
raw_score = qm.evaluate(raw_ctx)

md(f"{'Prompt':<35} {'Score'}")
md("-" * 50)
md(f"{'Raw (single generic prompt)':<35} {raw_score.overall * 100:.0f}/100")

for name, ctx in pipeline_contexts.items():
    s = qm.evaluate(ctx)
    md(f"{'Single: ' + name:<35} {s.overall * 100:.0f}/100")

if 'integrated' in dir():
    int_ctx = integrated.to_context()
    int_score = qm.evaluate(int_ctx)
    md(f"{'Integrated (5 templates)':<35} {int_score.overall * 100:.0f}/100  ← all five merged")


Prompt                              Score

--------------------------------------------------

Raw (single generic prompt)         22/100

Integrated (5 templates)            78/100  ← all five merged

## Step 6 — Export the integrated prompt for any framework

In [13]:
import json

if 'ctx' in dir():
    md("### OpenAI format")
    _sj = json.dumps(ctx.to_openai(), indent=2, default=str)
    md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")

    md("### CrewAI format")
    _sj = json.dumps(ctx.to_crewai(), indent=2, default=str)
    md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")

    md("### LangChain format")
    _sj = json.dumps(ctx.to_langchain(), indent=2, default=str)
    md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")
else:
    md("Run Steps 4–5 with an API key in `.env` to see framework exports.")
    md("Any `Context` supports `.to_openai()`, `.to_crewai()`, `.to_langchain()`, `.to_anthropic()`, `.to_autogen()`, …")


### OpenAI format

```json
{
  "messages": [
    {
      "role": "system",
      "content": "You are Mortgage Portfolio Risk and Impact Analyst.\n\nFollow these rules:\n1. Evaluate risks by likelihood and impact, focusing on strategic, operational, and financial categories.\n2. Assess downstream effects and consequences of rising interest rates on the mortgage portfolio.\n3. Prioritize risks and impacts by severity and likelihood.\n4. Develop pragmatic mitigation strategies for identified risks.\n5. Perform a risk-benefit analysis to inform decision-making.\n6. Provide clear, actionable recommendations for the board.\n\nCONSTRAINTS:\n\nMust include:\n  - Context Summary: Overview of rising interest rates and initial impact on mortgage portfolio.\n  - Risk Identification: List and categorize risks by strategic, operational, and financial impacts.\n  - Risk Analysis: Probability and impact assessment of identified risks.\n  - Impact Assessment: Downstream effects on the mortgage portfolio.\n  - Mitigation Strategies: Proposed actions to manage identified risks.\n  - Risk-Benefit Analysis: Evaluation of potential impacts versus benefits.\n  - Recommendations / Next Steps: Concrete actions for the board to consider.\n\n1. Summarize the current context of rising interest rates and its potential impact on the mortgage portfolio.\n2. Identify and categorize risks into strategic, operational, and financial categories, considering both direct and indirect effects.\n3. Analyze the probability and impact of each risk, prioritizing them by severity.\n4. Assess the downstream effects of these risks on the mortgage portfolio, including potential cascading and compounding impacts.\n5. Develop and propose mitigation strategies for the most significant risks.\n6. Conduct a risk-benefit analysis to weigh the potential impacts against the benefits of current strategies.\n7. Formulate clear recommendations and next steps for the board, focusing on risk management and strategic adjustments."
    }
  ],
  "temperature": 0.7,
  "max_tokens": 4096
}
```

### CrewAI format

```json
{
  "role": "Mortgage Portfolio Risk and Impact Analyst",
  "goal": "1. Summarize the current context of rising interest rates and its potential impact on the mortgage portfolio.\n2. Identify and categorize risks into strategic, operational, and financial categories, considering both direct and indirect effects.\n3. Analyze the probability and impact of each risk, prioritizing them by severity.\n4. Assess the downstream effects of these risks on the mortgage portfolio, including potential cascading and compounding impacts.\n5. Develop and propose mitigation strategies for the most significant risks.\n6. Conduct a risk-benefit analysis to weigh the potential impacts against the benefits of current strategies.\n7. Formulate clear recommendations and next steps for the board, focusing on risk management and strategic adjustments.",
  "backstory": "You are Mortgage Portfolio Risk and Impact Analyst.\n\nFollow these rules:\n1. Evaluate risks by likelihood and impact, focusing on strategic, operational, and financial categories.\n2. Assess downstream effects and consequences of rising interest rates on the mortgage portfolio.\n3. Prioritize risks and impacts by severity and likelihood.\n4. Develop pragmatic mitigation strategies for identified risks.\n5. Perform a risk-benefit analysis to inform decision-making.\n6. Provide clear, actionable recommendations for the board.",
  "context": "You are Mortgage Portfolio Risk and Impact Analyst.\n\nFollow these rules:\n1. Evaluate risks by likelihood and impact, focusing on strategic, operational, and financial categories.\n2. Assess downstream effects and consequences of rising interest rates on the mortgage portfolio.\n3. Prioritize risks and impacts by severity and likelihood.\n4. Develop pragmatic mitigation strategies for identified risks.\n5. Perform a risk-benefit analysis to inform decision-making.\n6. Provide clear, actionable recommendations for the board.\n\nCONSTRAINTS:\n\nMust include:\n  - Context Summary: Overview of rising interest rates and initial impact on mortgage portfolio.\n  - Risk Identification: List and categorize risks by strategic, operational, and financial impacts.\n  - Risk Analysis: Probability and impact assessment of identified risks.\n  - Impact Assessment: Downstream effects on the mortgage portfolio.\n  - Mitigation Strategies: Proposed actions to manage identified risks.\n  - Risk-Benefit Analysis: Evaluation of potential impacts versus benefits.\n  - Recommendations / Next Steps: Concrete actions for the board to consider.\n\n1. Summarize the current context of rising interest rates and its potential impact on the mortgage portfolio.\n2. Identify and categorize risks into strategic, operational, and financial categories, considering both direct and indirect effects.\n3. Analyze the probability and impact of each risk, prioritizing them by severity.\n4. Assess the downstream effects of these risks on the mortgage portfolio, including potential cascading and compounding impacts.\n5. Develop and propose mitigation strategies for the most significant risks.\n6. Conduct a risk-benefit analysis to weigh the potential impacts against the benefits of current strategies.\n7. Formulate clear recommendations and next steps for the board, focusing on risk management and strategic adjustments.",
  "expected_output": "Output must include: Context Summary: Overview of rising interest rates and initial impact on mortgage portfolio., Risk Identification: List and categorize risks by strategic, operational, and financial impacts., Risk Analysis: Probability and impact assessment of identified risks., Impact Assessment: Downstream effects on the mortgage portfolio., Mitigation Strategies: Proposed actions to manage identified risks., Risk-Benefit Analysis: Evaluation of potential impacts versus benefits., Recommendations / Next Steps: Concrete actions for the board to consider.",
  "tools": [],
  "verbose": true
}
```

### LangChain format

```json
{
  "system_message": "You are Mortgage Portfolio Risk and Impact Analyst.\n\nFollow these rules:\n1. Evaluate risks by likelihood and impact, focusing on strategic, operational, and financial categories.\n2. Assess downstream effects and consequences of rising interest rates on the mortgage portfolio.\n3. Prioritize risks and impacts by severity and likelihood.\n4. Develop pragmatic mitigation strategies for identified risks.\n5. Perform a risk-benefit analysis to inform decision-making.\n6. Provide clear, actionable recommendations for the board.\n\nCONSTRAINTS:\n\nMust include:\n  - Context Summary: Overview of rising interest rates and initial impact on mortgage portfolio.\n  - Risk Identification: List and categorize risks by strategic, operational, and financial impacts.\n  - Risk Analysis: Probability and impact assessment of identified risks.\n  - Impact Assessment: Downstream effects on the mortgage portfolio.\n  - Mitigation Strategies: Proposed actions to manage identified risks.\n  - Risk-Benefit Analysis: Evaluation of potential impacts versus benefits.\n  - Recommendations / Next Steps: Concrete actions for the board to consider.\n\n1. Summarize the current context of rising interest rates and its potential impact on the mortgage portfolio.\n2. Identify and categorize risks into strategic, operational, and financial categories, considering both direct and indirect effects.\n3. Analyze the probability and impact of each risk, prioritizing them by severity.\n4. Assess the downstream effects of these risks on the mortgage portfolio, including potential cascading and compounding impacts.\n5. Develop and propose mitigation strategies for the most significant risks.\n6. Conduct a risk-benefit analysis to weigh the potential impacts against the benefits of current strategies.\n7. Formulate clear recommendations and next steps for the board, focusing on risk management and strategic adjustments.",
  "context": {
    "guidance": {
      "role": "Mortgage Portfolio Risk and Impact Analyst",
      "rules": [
        "Evaluate risks by likelihood and impact, focusing on strategic, operational, and financial categories.",
        "Assess downstream effects and consequences of rising interest rates on the mortgage portfolio.",
        "Prioritize risks and impacts by severity and likelihood.",
        "Develop pragmatic mitigation strategies for identified risks.",
        "Perform a risk-benefit analysis to inform decision-making.",
        "Provide clear, actionable recommendations for the board."
      ],
      "style": null,
      "expertise": null,
      "goal": null,
      "persona_scope": null
    },
    "directive": {
      "content": "1. Summarize the current context of rising interest rates and its potential impact on the mortgage portfolio.\n2. Identify and categorize risks into strategic, operational, and financial categories, considering both direct and indirect effects.\n3. Analyze the probability and impact of each risk, prioritizing them by severity.\n4. Assess the downstream effects of these risks on the mortgage portfolio, including potential cascading and compounding impacts.\n5. Develop and propose mitigation strategies for the most significant risks.\n6. Conduct a risk-benefit analysis to weigh the potential impacts against the benefits of current strategies.\n7. Formulate clear recommendations and next steps for the board, focusing on risk management and strategic adjustments.",
      "priority": 5,
      "constraints": null,
      "tags": null
    },
    "constraints": {
      "must_include": [
        "Context Summary: Overview of rising interest rates and initial impact on mortgage portfolio.",
        "Risk Identification: List and categorize risks by strategic, operational, and financial impacts.",
        "Risk Analysis: Probability and impact assessment of identified risks.",
        "Impact Assessment: Downstream effects on the mortgage portfolio.",
        "Mitigation Strategies: Proposed actions to manage identified risks.",
        "Risk-Benefit Analysis: Evaluation of potential impacts versus benefits.",
        "Recommendations / Next Steps: Concrete actions for the board to consider."
      ],
      "must_not_include": null,
      "format_rules": null,
      "output_contract": null,
      "style_guide": null,
      "max_length": null,
      "language": null,
      "output_schema": null,
      "verbosity": null,
      "communication_posture": null,
      "forbidden_phrases": null,
      "answer_first": null,
      "self_check": null
    },
    "knowledge": null,
    "data": {
      "integration_metadata": {
        "source_templates": [
          "risk_assessor",
          "impact_assessor"
        ],
        "question": "How will rising interest rates affect our mortgage portfolio risk, and what should we tell the board?",
        "agent": "TemplateIntegratorAgent"
      }
    },
    "metadata": {},
    "thinking_strategy": null,
    "examples": null,
    "analytical_approach": null,
    "research_flow": false,
    "provider_hint": null,
    "task_contract": null
  },
  "guidance": {
    "role": "Mortgage Portfolio Risk and Impact Analyst",
    "rules": [
      "Evaluate risks by likelihood and impact, focusing on strategic, operational, and financial categories.",
      "Assess downstream effects and consequences of rising interest rates on the mortgage portfolio.",
      "Prioritize risks and impacts by severity and likelihood.",
      "Develop pragmatic mitigation strategies for identified risks.",
      "Perform a risk-benefit analysis to inform decision-making.",
      "Provide clear, actionable recommendations for the board."
    ],
    "style": null,
    "expertise": null,
    "goal": null,
    "persona_scope": null
  },
  "directive": {
    "content": "1. Summarize the current context of rising interest rates and its potential impact on the mortgage portfolio.\n2. Identify and categorize risks into strategic, operational, and financial categories, considering both direct and indirect effects.\n3. Analyze the probability and impact of each risk, prioritizing them by severity.\n4. Assess the downstream effects of these risks on the mortgage portfolio, including potential cascading and compounding impacts.\n5. Develop and propose mitigation strategies for the most significant risks.\n6. Conduct a risk-benefit analysis to weigh the potential impacts against the benefits of current strategies.\n7. Formulate clear recommendations and next steps for the board, focusing on risk management and strategic adjustments.",
    "priority": 5,
    "constraints": null,
    "tags": null
  },
  "knowledge": null
}
```

## Summary

| What you learned | API | Key? |
|-----------------|-----|------|
| Get full pipeline recommendation | `suggest_patterns(q, suggest_chain=True)` | No |
| Multiple route options with reasoning | `suggest_routes(q, max_routes=3)` | Yes |
| Quality-gate each template | `QualityMetrics(mode='heuristic').evaluate(ctx)` | No |
| Merge templates into one prompt | `TemplateIntegratorAgent().suggest_and_integrate(q)` | Yes |
| Export to any framework | `ctx.to_crewai()`, `.to_langchain()`, etc. | No |

**Next notebook:** [06_cross_framework.ipynb](06_cross_framework.ipynb) — one Context, ten framework exports, full agent creation helpers.